<a href="https://colab.research.google.com/github/vifirsanova/ML-2026-pt-2/blob/main/advanced/3_traditional_ml_in_mas/agents_libs_langgraph.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

## LangGraph

https://docs.langchain.com/oss/python/langgraph/workflows-agents

In [ ]:
!pip install langchain==0.3.0 langchain-core langchain_openai -q

In [ ]:
from langgraph.graph import StateGraph, END
from typing import TypedDict, List
from langchain_core.prompts import PromptTemplate
import os

In [ ]:
from langchain_openai import ChatOpenAI
from google.colab import userdata

llm = ChatOpenAI(
    api_key=userdata.get('OPEN_ROUTER_KEY'),
    base_url="https://openrouter.ai/api/v1",
    model="openrouter/aurora-alpha",
    #default_headers={
    #    "HTTP-Referer": getenv("YOUR_SITE_URL"),  # Optional. Site URL for rankings on openrouter.ai.
    #    "X-Title": getenv("YOUR_SITE_NAME"),  # Optional. Site title for rankings on openrouter.ai.
    #}
)

In [ ]:
response = llm.invoke("What NFL team won the Super Bowl in the year Justin Bieber was born?")
print(response.content)

Justin Bieber was born in 1994, and the Super Bowl played that calendar year was Super Bowl XXVIII (January 30 1994), which was won by the **Dallas Cowboys**.


In [ ]:
# Определение состояния
class AgentState(TypedDict):
    messages: List[str]
    current_task: str
    result: str

# Создание узлов графа
def research_node(state: AgentState):
    prompt = PromptTemplate.from_template(
        "Summarize literature on {topic}"
    )
    chain = prompt | llm
    result = chain.invoke({"topic": state["current_task"]})

    return {
        "messages": state["messages"] + [f"Research: {result.content}"],
        "result": result.content
    }

def write_node(state: AgentState):
    prompt = PromptTemplate.from_template(
        "Prepare a research paper plan based on the literature study: {research}"
    )
    chain = prompt | llm
    result = chain.invoke({"research": state["result"]})

    return {
        "messages": state["messages"] + [f"Article: {result.content}"],
        "result": result.content
    }

# Построение графа
workflow = StateGraph(AgentState)
workflow.add_node("research", research_node)
workflow.add_node("write", write_node)

workflow.set_entry_point("research")
workflow.add_edge("research", "write")
workflow.add_edge("write", END)

# Компиляция и запуск
app = workflow.compile()

result = app.invoke({
    "messages": [],
    "current_task": "Искусственный интеллект в медицине",
    "result": ""
})

print(result["result"])

**План исследовательской статьи**  
*Тема*: **Искусственный интеллект в медицине: обзор ключевой литературы (русскоязычные и международные источники) и перспективы дальнейшего развития**  

---

## 1. Введение  

| Параметр | Содержание |
|----------|------------|
| **Актуальность** | За последние 10 лет AI‑технологии перешли от экспериментальных прототипов к сертифицированным медицинским продуктам, получившим одобрение регуляторов (FDA, EU AI Act) и внедрённым в крупные клинические сети. |
| **Проблематика** | Несмотря на значительные успехи, остаются открытыми вопросы качества данных, объяснимости, этичности, регулятивных барьеров и интеграции в клинический workflow. |
| **Цель исследования** | Сформулировать системный обзор литературы по применению ИИ в медицине, выявить главные тенденции, технологический прогресс и пробелы, а также предложить направления будущих исследований и практического внедрения. |
| **Задачи** | 1) Классифицировать основные направления применения ИИ (диагност

In [ ]:
print(result['messages'][0])

Research: **Искусственный интеллект в медицине: обзор ключевой литературы (русскоязычные и международные источники)**  

---

## 1. Общие тенденции и рамки исследования  

| Год | Основные публикации (англ./рус.) | Краткое содержание |
|-----|--------------------------------|--------------------|
| **2015‑2017** | *Topol J. E. Artificial Intelligence in Medicine* (Nature Medicine); *Баранов А. и др. «Искусственный интеллект в здравоохранении»* (Медицинская информатика) | Описывают первые практические примеры AI‑систем в радиологии и патологии, поднимают вопросы интеграции в клиническую практику. |
| **2018‑2020** | *Esteva A. et al. “A guide learning algorithm for diagnosis of skin cancer”* (Nature); *Кузнецов В. и др. «Машинное обучение в клинической диагностике»* (Журнал клинической медицины) | Показали, что глубокие нейронные сети способны достигать уровня экспертов в задачах визуального распознавания (кожные опухоли, ретинальные изображения). |
| **2021‑2023** | *McKinsey & Company

In [ ]:
print(result['messages'][1])

Article: **План исследовательской статьи**  
*Тема*: **Искусственный интеллект в медицине: обзор ключевой литературы (русскоязычные и международные источники) и перспективы дальнейшего развития**  

---

## 1. Введение  

| Параметр | Содержание |
|----------|------------|
| **Актуальность** | За последние 10 лет AI‑технологии перешли от экспериментальных прототипов к сертифицированным медицинским продуктам, получившим одобрение регуляторов (FDA, EU AI Act) и внедрённым в крупные клинические сети. |
| **Проблематика** | Несмотря на значительные успехи, остаются открытыми вопросы качества данных, объяснимости, этичности, регулятивных барьеров и интеграции в клинический workflow. |
| **Цель исследования** | Сформулировать системный обзор литературы по применению ИИ в медицине, выявить главные тенденции, технологический прогресс и пробелы, а также предложить направления будущих исследований и практического внедрения. |
| **Задачи** | 1) Классифицировать основные направления применения ИИ 